In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import sys
from pathlib import Path

# Поменяй путь под свою папку на Google Drive.
# В этой папке должны лежать: dataset.py, model.py, trainer.py.
PROJECT_DIR = Path('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training')
DATA_DIR = PROJECT_DIR / 'data' / 'gold_labeled_data'

os.chdir(PROJECT_DIR)

In [3]:
import gc
import json
import random
import warnings
from datetime import datetime
from typing import Optional
import re
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from jinja2 import Template
from tqdm.auto import tqdm


warnings.filterwarnings('ignore')

RANDOM_SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA memory, GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA memory, GB: 94.97


In [4]:
# !pip install -q bert-score

In [5]:
# %pip install --upgrade transformers

# Грузим датасеты

In [6]:
FULL_DATA_XLSX = DATA_DIR / 'app_reviews_ru_gold_labeled.xlsx'
TRAIN_XLSX = DATA_DIR / 'train.xlsx'
TEST_XLSX = DATA_DIR / 'test.xlsx'

DROP_COLS = ['Unnamed: 0', 'Unnamed: 0.1', 'lable_silver_common_AI', 'summary_silver_GPT']


def load_review_data():
    if TRAIN_XLSX.exists() and TEST_XLSX.exists():
        print('Loading existing train.xlsx/test.xlsx')
        train_df = pd.read_excel(TRAIN_XLSX)
        test_df = pd.read_excel(TEST_XLSX)
    else:
        print('train.xlsx/test.xlsx were not found, creating stratified split from FINAL_GOLD')
        df = pd.read_excel(FULL_DATA_XLSX, sheet_name='FINAL_GOLD')
        df = df.drop(columns=DROP_COLS, errors='ignore')
        df = df.dropna(subset=['text', 'label_gold']).reset_index(drop=True)

        train_df, test_df = train_test_split(
            df,
            test_size=0.15,
            random_state=RANDOM_SEED,
            stratify=df['label_gold']
        )

        train_df = train_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)
        train_df.to_excel(TRAIN_XLSX, index=False)
        test_df.to_excel(TEST_XLSX, index=False)

    train_df = train_df.drop(columns=DROP_COLS, errors='ignore')
    test_df = test_df.drop(columns=DROP_COLS, errors='ignore')

    for df in [train_df, test_df]:
        df['text'] = df['text'].fillna('').astype(str)
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df['thumbs_up_count'] = pd.to_numeric(df['thumbs_up_count'], errors='coerce')

    return train_df, test_df

train_data, test_data = load_review_data()

print('Train:', train_data.shape)
print('Test:', test_data.shape)
print('Train columns:', train_data.columns.tolist())
print('Class distribution:')
print(train_data['label_gold'].value_counts(normalize=True).round(4))
train_data.head()

Loading existing train.xlsx/test.xlsx
Train: (2714, 6)
Test: (480, 6)
Train columns: ['app_name', 'text', 'rating', 'thumbs_up_count', 'label_gold', 'summary_gold']
Class distribution:
label_gold
user_experience    0.4749
bug_report         0.2598
rating             0.2119
feature_request    0.0534
Name: proportion, dtype: float64


,app_name,text,rating,thumbs_up_count,label_gold,summary_gold
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает


# System prompt

In [ ]:
system_prompt = Template(
"""
Ты — NLP-ассистент для анализа русскоязычных отзывов на приложения Google Play.

Тебе будет передан один отзыв в формате:
text: текст отзыва
rating: оценка приложения от 1 до 5
thumbs_up_count: количество лайков отзыва

Твоя задача:
1. Определить основной класс отзыва строго из списка:
- bug_report
- feature_request
- user_experience
- rating

Правила разметки:
- bug_report: пользователь сообщает об ошибке, сбое, зависании, некорректной работе, потере данных.
- feature_request: пользователь просит добавить или изменить функцию. Если одновременно есть явная ошибка и просьба, выбери bug_report.
- user_experience: пользователь описывает опыт использования, удобство, неудобство, плюсы/минусы без явного бага и без просьбы новой функции.
- rating: короткая общая оценка без полезной конкретики.

Приоритет при смешанных случаях:
bug_report > feature_request > user_experience > rating

2. Сформировать summary:
- одна короткая фраза на русском;
- не более 12 слов;
- без выдуманных деталей;
- только главная мысль из отзыва.

Верни только валидный JSON без markdown и пояснений:
{"label": "bug_report|feature_request|user_experience|rating", "summary": "короткое summary"}

Примеры:
Отзыв:
text: все приложение не работает вообще, не заходит и не грузит
rating: 1
thumbs_up_count: 0
Ответ:
{"label": "bug_report", "summary": "Приложение не запускается и не загружается"}

Отзыв:
text: отлично приложение!!!
rating: 5
thumbs_up_count: 0
Ответ:
{"label": "rating", "summary": "Краткая положительная оценка без деталей"}

Отзыв:
text: прошу добавить темную тему
rating: 4
thumbs_up_count: 2
Ответ:
{"label": "feature_request", "summary": "Просьба добавить темную тему"}
"""
)


In [19]:
# Утилиты для инференса LLM

VALID_LABELS = {"bug_report", "feature_request", "user_experience", "rating"}


def format_review(row_or_text, rating=None, thumbs_up_count=None) -> str:
    """
    Приводит отзыв к тому формату, который описан в system_prompt.
    Можно передать либо строку, либо строку DataFrame.
    """
    if isinstance(row_or_text, pd.Series):
        text = str(row_or_text.get("text", ""))
        rating = row_or_text.get("rating", "")
        thumbs_up_count = row_or_text.get("thumbs_up_count", "")
    else:
        text = str(row_or_text)

    return (
        f"text: {text}\n"
        f"rating: {rating if rating is not None else ''}\n"
        f"thumbs_up_count: {thumbs_up_count if thumbs_up_count is not None else ''}"
    )


def prepare_messages(review_text: str, system_prompt_template: Template) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt_template.render()},
        {"role": "user", "content": review_text},
    ]


def merge_system_into_user(messages):
    """
    Fallback для моделей/токенизаторов, которые не поддерживают отдельную role="system".
    Например, часть Mistral-шаблонов ожидает только user/assistant.
    В таком случае system_prompt аккуратно добавляется в первый user-message.
    """
    system_parts = []
    user_parts = []

    for msg in messages:
        role = msg.get("role", "")
        content = str(msg.get("content", ""))

        if role == "system":
            system_parts.append(content)
        elif role == "user":
            user_parts.append(content)
        else:
            # Для текущей задачи assistant-сообщений обычно нет, но оставим безопасную обработку.
            user_parts.append(content)

    system_text = "\n\n".join(system_parts).strip()
    user_text = "\n\n".join(user_parts).strip()

    if system_text:
        merged_content = (
            "Инструкция:\n"
            f"{system_text}\n\n"
            "Входные данные:\n"
            f"{user_text}"
        )
    else:
        merged_content = user_text

    return [{"role": "user", "content": merged_content}]


def apply_chat_template_safe(tokenizer, messages, enable_thinking: bool):
    """
    Вспомогательная функция, чтобы не дублировать параметры apply_chat_template.
    """
    kwargs = dict(
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    if enable_thinking:
        kwargs["enable_thinking"] = False

    return tokenizer.apply_chat_template(messages, **kwargs)


def build_chat_inputs(tokenizer, messages):
    """
    Универсальная подготовка входа для разных chat-моделей.

    У Qwen3 можно передавать enable_thinking=False, чтобы отключить reasoning.
    У Phi/Mistral/Llama такого аргумента может не быть, поэтому делаем fallback.
    У некоторых Mistral-шаблонов может не поддерживаться отдельная role="system",
    поэтому при ошибке system_prompt объединяется с user-сообщением.
    """
    # 1) Пробуем обычный chat-template с system/user.
    try:
        return apply_chat_template_safe(tokenizer, messages, enable_thinking=True)
    except TypeError:
        # Модель не знает аргумент enable_thinking.
        try:
            return apply_chat_template_safe(tokenizer, messages, enable_thinking=False)
        except Exception:
            pass
    except Exception:
        pass

    # 2) Fallback: объединяем system_prompt с user-сообщением.
    merged_messages = merge_system_into_user(messages)

    try:
        return apply_chat_template_safe(tokenizer, merged_messages, enable_thinking=True)
    except TypeError:
        return apply_chat_template_safe(tokenizer, merged_messages, enable_thinking=False)

def generate_response(model, tokenizer, messages, max_new_tokens=96):
    """
    Один вызов генерации для одного отзыва.
    Ответ должен быть коротким JSON, поэтому max_new_tokens оставляем маленьким.
    """
    model.eval()

    inputs = build_chat_inputs(tokenizer, messages)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            use_cache=True,
        )

    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()


def parse_llm_json(response: str) -> dict:
    """
    Пытается достать JSON из ответа модели.
    Если модель добавила лишний текст, вырезаем первый {...}.
    """
    raw_response = str(response)

    response = re.sub(r"<think>.*?</think>", "", raw_response, flags=re.DOTALL).strip()
    response = response.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*?\}", response, flags=re.DOTALL)
    if match:
        response = match.group(0)

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        return {
            "label": "parse_error",
            "summary": response[:120],
            "raw_response": raw_response,
        }

    label = str(data.get("label", "")).strip()
    summary = str(data.get("summary", "")).strip()

    if label not in VALID_LABELS:
        return {
            "label": "parse_error",
            "summary": summary[:120],
            "raw_response": raw_response,
        }

    # Жёстко ограничим summary на случай, если модель разговорилась
    summary = " ".join(summary.split()[:12])

    return {
        "label": label,
        "summary": summary,
        "raw_response": raw_response,
    }


In [ ]:
def run_agent(row_or_text, model, tokenizer, system_prompt: Template, debug: bool = False):
    """
    Один отзыв -> один вызов model.generate -> dict с label и summary.
    Старый while True здесь не нужен, потому что tools/function calling не используются.
    """
    review_text = format_review(row_or_text)
    messages = prepare_messages(review_text, system_prompt)

    if debug:
        print("USER MESSAGE:")
        print(messages[-1]["content"])

    response = generate_response(
        model=model,
        tokenizer=tokenizer,
        messages=messages,
        max_new_tokens=96,
    )

    if debug:
        print("\nRAW MODEL RESPONSE:")
        print(response)

    return parse_llm_json(response)


# Инференс и сравнение 4 LLM

Ниже идут четыре независимых блока в одном стиле:

1. загрузка модели;
2. инференс по `test_data`;
3. сохранение отдельного `res_df` в CSV;
4. очистка модели из памяти.

Выбранные модели:

| Группа | Модель | Зачем сравниваем |
|---|---|---|
| ~4B | `Qwen/Qwen3-4B-Instruct-2507` | сильная современная multilingual instruct-модель на 4B |
| ~4B | `microsoft/Phi-4-mini-instruct` | компактный внешний конкурент примерно на 3.8B |
| 7B | `Qwen/Qwen2.5-7B-Instruct` | сильная 7B multilingual-модель с явной поддержкой русского |
| 7B | `mistralai/Mistral-7B-Instruct-v0.3` | сильный внешний 7B-конкурент другого семейства; интересен для LoRA/QLoRA |

Логика выбора: в сравнении остаются две модели около 4B и две модели около 7B, но теперь 7B-группа сравнивает не только размер, а ещё и разные семейства моделей: Qwen против Mistral.


In [14]:
# При необходимости раскомментируй установку/обновление зависимостей.
# Для 4-bit загрузки нужен bitsandbytes.
# %pip install -q -U transformers accelerate bitsandbytes

from transformers import BitsAndBytesConfig

# Лучше сохранять CSV прямо в папку проекта, чтобы они не потерялись после перезапуска Colab.
RESULTS_DIR = PROJECT_DIR / "LLM_Inference" / "llm_inference_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Для Colab/T4/G4 безопаснее включить 4-bit, особенно для 7B/8B моделей.
# Если у тебя A100/H100 и хочешь честный FP16-инференс, поставь LOAD_IN_4BIT = False.
LOAD_IN_4BIT = False

bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU memory allocated: {allocated:.2f} GB")
        print(f"GPU memory reserved:  {reserved:.2f} GB")

print("RESULTS_DIR:", RESULTS_DIR)
print("LOAD_IN_4BIT:", LOAD_IN_4BIT)
print_gpu_memory()


RESULTS_DIR: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_Inference/llm_inference_results
LOAD_IN_4BIT: False
GPU memory allocated: 0.00 GB
GPU memory reserved:  0.00 GB


## Qwen3-4B-Instruct-2507

In [ ]:
# ==============================
# Модель: Qwen3-4B-Instruct-2507
# ==============================

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
}

if LOAD_IN_4BIT:
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_kwargs,
)

model.eval()
print("Loaded:", model_name)
print_gpu_memory()


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-4B-Instruct-2507
GPU memory allocated: 7.49 GB
GPU memory reserved:  7.49 GB


In [ ]:
# Инференс по всей тестовой выборке + classification_report
# Модель: Qwen3-4B-Instruct-2507

pred_rows = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data)):
    pred = run_agent(row, model, tokenizer, system_prompt, debug=False)

    pred_rows.append({
        "id": idx,
        "text": row["text"],
        "rating": row.get("rating", None),
        "thumbs_up_count": row.get("thumbs_up_count", None),
        "label_gold": row.get("label_gold", None),
        "summary_gold": row.get("summary_gold", None),
        "label_pred": pred["label"],
        "summary_pred": pred["summary"],
        "raw_response": pred.get("raw_response", ""),
    })

res_qwen3_4b_instruct_2507 = pd.DataFrame(pred_rows)
res_df = res_qwen3_4b_instruct_2507.copy()

# Отдельно посмотрим ошибки парсинга, если они есть
print("Parse errors:", (res_df["label_pred"] == "parse_error").sum())

# Метрики по классификации
valid_eval = res_df[res_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

print(classification_report(
    valid_eval["label_gold"],
    valid_eval["label_pred"],
    labels=["bug_report", "feature_request", "user_experience", "rating"],
    digits=4,
    zero_division=0,
))

csv_path = RESULTS_DIR / "qwen3_4b_instruct_2507_predictions.csv"
res_df.to_csv(csv_path, index=False)

print("Saved to:", csv_path)
res_df.head()


  0%|          | 0/480 [00:00<?, ?it/s]

Parse errors: 1
                 precision    recall  f1-score   support

     bug_report     0.6069    0.8400    0.7047       125
feature_request     0.5000    0.5385    0.5185        26
user_experience     0.7925    0.5551    0.6528       227
         rating     0.7479    0.8812    0.8091       101

       accuracy                         0.6973       479
      macro avg     0.6618    0.7037    0.6713       479
   weighted avg     0.7188    0.6973    0.6920       479

Saved to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference/llm_inference_results/qwen3_4b_instruct_2507_predictions.csv


,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,Сообщения не доходят и не видны,"{""label"": ""bug_report"", ""summary"": ""Сообщения ..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,bug_report,После удаления историй нельзя публиковать,"{""label"": ""bug_report"", ""summary"": ""После удал..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,Приложение виснит и вылетает при музыке,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Приложение после обновления не работает,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,user_experience,"Неудобное приложение, проблемы с использованием","{""label"": ""user_experience"", ""summary"": ""Неудо..."


In [ ]:
res_df[res_df["label_pred"] == "parse_error"]

,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
33,33,"Авито ужасно скатились, теперь объявление выло...",1,0,user_experience,Недовольство платными функциями,parse_error,"Приложение лагает, объявления платные, много м...","{""label"": ""bug_report|user_experience"", ""summa..."


In [ ]:
# Удаляем модель из памяти после инференса: Qwen3-4B-Instruct-2507

del model
del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Model deleted:", "Qwen3-4B-Instruct-2507")
print_gpu_memory()


Model deleted: Qwen3-4B-Instruct-2507
GPU memory allocated: 0.01 GB
GPU memory reserved:  0.02 GB


## Phi-4-mini-instruct

In [ ]:
# ==============================
# Модель: Phi-4-mini-instruct
# ==============================

model_name = "microsoft/Phi-4-mini-instruct"

# На всякий случай очищаем старый remote-code кэш Phi,
# потому что именно cached modeling_phi3.py часто вызывает LossKwargs error.
!rm -rf ~/.cache/huggingface/modules/transformers_modules/microsoft/Phi-4-mini-instruct
!rm -rf ~/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=False,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": False,
}

if LOAD_IN_4BIT:
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_kwargs,
)

model.eval()
print("Loaded:", model_name)
print_gpu_memory()

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Loaded: microsoft/Phi-4-mini-instruct
GPU memory allocated: 7.15 GB
GPU memory reserved:  7.18 GB


In [ ]:
# Инференс по всей тестовой выборке + classification_report
# Модель: Phi-4-mini-instruct

pred_rows = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data)):
    pred = run_agent(row, model, tokenizer, system_prompt, debug=False)

    pred_rows.append({
        "id": idx,
        "text": row["text"],
        "rating": row.get("rating", None),
        "thumbs_up_count": row.get("thumbs_up_count", None),
        "label_gold": row.get("label_gold", None),
        "summary_gold": row.get("summary_gold", None),
        "label_pred": pred["label"],
        "summary_pred": pred["summary"],
        "raw_response": pred.get("raw_response", ""),
    })

res_phi4_mini_instruct = pd.DataFrame(pred_rows)
res_df = res_phi4_mini_instruct.copy()

# Отдельно посмотрим ошибки парсинга, если они есть
print("Parse errors:", (res_df["label_pred"] == "parse_error").sum())

# Метрики по классификации
valid_eval = res_df[res_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

print(classification_report(
    valid_eval["label_gold"],
    valid_eval["label_pred"],
    labels=["bug_report", "feature_request", "user_experience", "rating"],
    digits=4,
    zero_division=0,
))

csv_path = RESULTS_DIR / "phi4_mini_instruct_predictions.csv"
res_df.to_csv(csv_path, index=False)

print("Saved to:", csv_path)
res_df.head()


  0%|          | 0/480 [00:00<?, ?it/s]

Parse errors: 0
                 precision    recall  f1-score   support

     bug_report     0.5161    0.8960    0.6550       125
feature_request     0.5500    0.4231    0.4783        26
user_experience     0.6872    0.5877    0.6336       228
         rating     0.9375    0.4455    0.6040       101

       accuracy                         0.6292       480
      macro avg     0.6727    0.5881    0.5927       480
   weighted avg     0.6879    0.6292    0.6245       480

Saved to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference/llm_inference_results/phi4_mini_instruct_predictions.csv


,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,"Сообщения не доходят, приложение не работает","{""label"": ""bug_report"", ""summary"": ""Сообщения ..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,bug_report,Ошибка при публикации после бесплатных историй,"{""label"": ""bug_report"", ""summary"": ""Ошибка при..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,Приложение виснит при воспроизведении музыки,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Обновление привело к сбою приложения,"{""label"": ""bug_report"", ""summary"": ""Обновление..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,bug_report,"Приложение неудобное, проблемы с работой, треб...","{""label"": ""bug_report"", ""summary"": ""Приложение..."


In [ ]:
# Удаляем модель из памяти после инференса: Phi-4-mini-instruct

del model
del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Model deleted:", "Phi-4-mini-instruct")
print_gpu_memory()


Model deleted: Phi-4-mini-instruct
GPU memory allocated: 0.01 GB
GPU memory reserved:  0.02 GB


## Qwen2.5-7B-Instruct

In [ ]:
# ==============================
# Модель: Qwen2.5-7B-Instruct
# ==============================

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
}

if LOAD_IN_4BIT:
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_kwargs,
)

model.eval()
print("Loaded:", model_name)
print_gpu_memory()


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct
GPU memory allocated: 14.19 GB
GPU memory reserved:  14.22 GB


In [ ]:
# Инференс по всей тестовой выборке + classification_report
# Модель: Qwen2.5-7B-Instruct

pred_rows = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data)):
    pred = run_agent(row, model, tokenizer, system_prompt, debug=False)

    pred_rows.append({
        "id": idx,
        "text": row["text"],
        "rating": row.get("rating", None),
        "thumbs_up_count": row.get("thumbs_up_count", None),
        "label_gold": row.get("label_gold", None),
        "summary_gold": row.get("summary_gold", None),
        "label_pred": pred["label"],
        "summary_pred": pred["summary"],
        "raw_response": pred.get("raw_response", ""),
    })

res_qwen25_7b_instruct = pd.DataFrame(pred_rows)
res_df = res_qwen25_7b_instruct.copy()

# Отдельно посмотрим ошибки парсинга, если они есть
print("Parse errors:", (res_df["label_pred"] == "parse_error").sum())

# Метрики по классификации
valid_eval = res_df[res_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

print(classification_report(
    valid_eval["label_gold"],
    valid_eval["label_pred"],
    labels=["bug_report", "feature_request", "user_experience", "rating"],
    digits=4,
    zero_division=0,
))

csv_path = RESULTS_DIR / "qwen25_7b_instruct_predictions.csv"
res_df.to_csv(csv_path, index=False)

print("Saved to:", csv_path)
res_df.head()


  0%|          | 0/480 [00:00<?, ?it/s]

Parse errors: 1
                 precision    recall  f1-score   support

     bug_report     0.4384    0.9758    0.6050       124
feature_request     0.6471    0.4231    0.5116        26
user_experience     0.8548    0.2325    0.3655       228
         rating     0.7016    0.8614    0.7733       101

       accuracy                         0.5678       479
      macro avg     0.6605    0.6232    0.5639       479
   weighted avg     0.7034    0.5678    0.5214       479

Saved to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference/llm_inference_results/qwen25_7b_instruct_predictions.csv


,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,Обновления делают приложение хуже и неработосп...,"{""label"": ""bug_report"", ""summary"": ""Обновления..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,bug_report,Ошибка при публикации историй после удаления,"{""label"": ""bug_report"", ""summary"": ""Ошибка при..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,Приложение вылетает при включении музыки,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Приложение перестало работать после обновления,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,bug_report,Много багов и запрет на использование Katemobile,"{""label"": ""bug_report"", ""summary"": ""Много баго..."


In [ ]:
# Удаляем модель из памяти после инференса: Qwen2.5-7B-Instruct

del model
del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Model deleted:", "Qwen2.5-7B-Instruct")
print_gpu_memory()


Model deleted: Qwen2.5-7B-Instruct
GPU memory allocated: 0.01 GB
GPU memory reserved:  0.02 GB


## Mistral-7B-Instruct-v0.3

Если Hugging Face попросит доступ к модели, нужно предварительно авторизоваться через `huggingface_hub.login()` и принять условия модели на странице Hugging Face.

In [ ]:
# ==============================
# Модель: Mistral-7B-Instruct-v0.3
# ==============================

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
}

if LOAD_IN_4BIT:
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_kwargs,
)

model.eval()
print("Loaded:", model_name)
print_gpu_memory()


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.3
GPU memory allocated: 13.51 GB
GPU memory reserved:  13.53 GB


In [ ]:
# Инференс по всей тестовой выборке + classification_report
# Модель: Mistral-7B-Instruct-v0.3

pred_rows = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data)):
    pred = run_agent(row, model, tokenizer, system_prompt, debug=False)

    pred_rows.append({
        "id": idx,
        "text": row["text"],
        "rating": row.get("rating", None),
        "thumbs_up_count": row.get("thumbs_up_count", None),
        "label_gold": row.get("label_gold", None),
        "summary_gold": row.get("summary_gold", None),
        "label_pred": pred["label"],
        "summary_pred": pred["summary"],
        "raw_response": pred.get("raw_response", ""),
    })

res_mistral_7b_instruct_v03 = pd.DataFrame(pred_rows)
res_df = res_mistral_7b_instruct_v03.copy()

# Отдельно посмотрим ошибки парсинга, если они есть
print("Parse errors:", (res_df["label_pred"] == "parse_error").sum())

# Метрики по классификации
valid_eval = res_df[res_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

print(classification_report(
    valid_eval["label_gold"],
    valid_eval["label_pred"],
    labels=["bug_report", "feature_request", "user_experience", "rating"],
    digits=4,
    zero_division=0,
))

csv_path = RESULTS_DIR / "mistral_7b_instruct_v03_predictions.csv"
res_df.to_csv(csv_path, index=False)

print("Saved to:", csv_path)
res_df.head()


  0%|          | 0/480 [00:00<?, ?it/s]

Parse errors: 2
                 precision    recall  f1-score   support

     bug_report     0.5112    0.9120    0.6552       125
feature_request     0.7500    0.2400    0.3636        25
user_experience     0.6476    0.5991    0.6224       227
         rating     0.9730    0.3564    0.5217       101

       accuracy                         0.6109       478
      macro avg     0.7205    0.5269    0.5407       478
   weighted avg     0.6860    0.6109    0.5962       478

Saved to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference/llm_inference_results/mistral_7b_instruct_v03_predictions.csv


,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,Не доставляются сообщения,"{""label"": ""bug_report"", ""summary"": ""Не доставл..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,user_experience,"Пользователь описывает удобство приложения, но...","{""label"": ""user_experience"", ""summary"": ""Польз..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,При включении музыки постоянно виснет и вылета...,"{""label"": ""bug_report"", ""summary"": ""При включе..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Приложение не работает после обновления,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,user_experience,"Приложение неудобно, имеются проблемы, пользов...","{""label"": ""user_experience"", ""summary"": ""Прило..."


In [ ]:
# Удаляем модель из памяти после инференса: Mistral-7B-Instruct-v0.3

del model
del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Model deleted:", "Mistral-7B-Instruct-v0.3")
print_gpu_memory()


Model deleted: Mistral-7B-Instruct-v0.3
GPU memory allocated: 0.01 GB
GPU memory reserved:  0.02 GB


In [17]:
# Список сохранённых файлов с предсказаниями

prediction_files = {
    "Qwen3-4B-Instruct-2507": RESULTS_DIR / "qwen3_4b_instruct_2507_predictions.csv",
    "Phi-4-mini-instruct": RESULTS_DIR / "phi4_mini_instruct_predictions.csv",
    "Qwen2.5-7B-Instruct": RESULTS_DIR / "qwen25_7b_instruct_predictions.csv",
    "Mistral-7B-Instruct-v0.3": RESULTS_DIR / "mistral_7b_instruct_v03_predictions.csv",
}

prediction_files


{'Qwen3-4B-Instruct-2507': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_Inference/llm_inference_results/qwen3_4b_instruct_2507_predictions.csv'),
 'Phi-4-mini-instruct': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_Inference/llm_inference_results/phi4_mini_instruct_predictions.csv'),
 'Qwen2.5-7B-Instruct': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_Inference/llm_inference_results/qwen25_7b_instruct_predictions.csv'),
 'Mistral-7B-Instruct-v0.3': PosixPath('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference_and_Training/LLM_Inference/llm_inference_results/mistral_7b_instruct_v03_predictions.csv')}

In [22]:
import re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from bert_score import score as bert_score


def normalize_text(text):
    """
    Нормализация текста summary:
    - приводим к строке;
    - убираем лишние пробелы;
    - приводим к нижнему регистру;
    - заменяем ё на е.
    """
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()
    text = text.replace("ё", "е")
    text = re.sub(r"\s+", " ", text)

    return text


def tokenize_ru(text):
    """
    Простая токенизация для русскоязычного текста.
    Берем русские/английские слова и числа.
    """
    text = normalize_text(text)
    return re.findall(r"[а-яa-z0-9]+", text)


def lcs_length(tokens_a, tokens_b):
    """
    Длина наибольшей общей подпоследовательности.
    Нужна для ROUGE-L.
    """
    n = len(tokens_a)
    m = len(tokens_b)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n):
        for j in range(m):
            if tokens_a[i] == tokens_b[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[n][m]


def rouge_l_score(reference, prediction):
    """
    Считает ROUGE-L Precision, Recall и F1 для одной пары:
    reference = summary_gold
    prediction = summary_pred
    """
    ref_tokens = tokenize_ru(reference)
    pred_tokens = tokenize_ru(prediction)

    if len(ref_tokens) == 0 and len(pred_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0, 0.0, 0.0

    lcs = lcs_length(ref_tokens, pred_tokens)

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


def evaluate_summaries(
    df,
    gold_col="summary_gold",
    pred_col="summary_pred",
    label_col=None,
    bert_model="xlm-roberta-base",
    batch_size=16
):
    """
    Оценивает качество summary по:
    - ROUGE-L;
    - BERTScore.

    Возвращает:
    - detailed_df: исходный датафрейм + метрики для каждой строки;
    - overall_metrics: общие средние метрики;
    - by_label_metrics: метрики по классам, если передан label_col.
    """

    detailed_df = df.copy()

    detailed_df[gold_col] = detailed_df[gold_col].fillna("").astype(str)
    detailed_df[pred_col] = detailed_df[pred_col].fillna("").astype(str)

    # ---------- ROUGE-L ----------
    rouge_p_list = []
    rouge_r_list = []
    rouge_f1_list = []

    for gold, pred in tqdm(
        zip(detailed_df[gold_col], detailed_df[pred_col]),
        total=len(detailed_df),
        desc="Calculating ROUGE-L"
    ):
        p, r, f1 = rouge_l_score(gold, pred)
        rouge_p_list.append(p)
        rouge_r_list.append(r)
        rouge_f1_list.append(f1)

    detailed_df["rougeL_precision"] = rouge_p_list
    detailed_df["rougeL_recall"] = rouge_r_list
    detailed_df["rougeL_f1"] = rouge_f1_list

    # ---------- BERTScore ----------
    gold_texts = detailed_df[gold_col].apply(normalize_text).tolist()
    pred_texts = detailed_df[pred_col].apply(normalize_text).tolist()

    valid_mask = [
        len(gold.strip()) > 0 and len(pred.strip()) > 0
        for gold, pred in zip(gold_texts, pred_texts)
    ]

    detailed_df["bertscore_precision"] = 0.0
    detailed_df["bertscore_recall"] = 0.0
    detailed_df["bertscore_f1"] = 0.0

    valid_gold = [gold for gold, is_valid in zip(gold_texts, valid_mask) if is_valid]
    valid_pred = [pred for pred, is_valid in zip(pred_texts, valid_mask) if is_valid]

    if len(valid_gold) > 0:
        device = "cuda" if torch.cuda.is_available() else "cpu"

        P, R, F1 = bert_score(
            cands=valid_pred,
            refs=valid_gold,
            model_type=bert_model,
            batch_size=batch_size,
            device=device,
            verbose=True
        )

        valid_indices = detailed_df.index[valid_mask]

        detailed_df.loc[valid_indices, "bertscore_precision"] = P.cpu().numpy()
        detailed_df.loc[valid_indices, "bertscore_recall"] = R.cpu().numpy()
        detailed_df.loc[valid_indices, "bertscore_f1"] = F1.cpu().numpy()

    # ---------- Общие метрики ----------
    metric_cols = [
        "rougeL_precision",
        "rougeL_recall",
        "rougeL_f1",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1"
    ]

    overall_metrics = detailed_df[metric_cols].mean().to_frame("mean_value")

    # ---------- Метрики по классам ----------
    by_label_metrics = None

    if label_col is not None and label_col in detailed_df.columns:
        by_label_metrics = (
            detailed_df
            .groupby(label_col)[metric_cols]
            .mean()
            .sort_values("bertscore_f1", ascending=False)
        )

    return detailed_df, overall_metrics, by_label_metrics

# Сравнение метрик по сохранённым CSV

Эта ячейка читает сохранённые файлы предсказаний и собирает одну таблицу по классификации и summary-метрикам.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score


def evaluate_saved_prediction_file(model_display_name, csv_path):
    df = pd.read_csv(csv_path)

    # Убираем parse_error из классификационных метрик.
    valid_cls = df[df["label_pred"].isin(sorted(VALID_LABELS))].copy()

    cls_metrics = {
        "model": model_display_name,
        "n_rows": len(df),
        "parse_errors": int((df["label_pred"] == "parse_error").sum()),
        "accuracy": accuracy_score(valid_cls["label_gold"], valid_cls["label_pred"]),
        "macro_f1": f1_score(
            valid_cls["label_gold"],
            valid_cls["label_pred"],
            labels=["bug_report", "feature_request", "user_experience", "rating"],
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            valid_cls["label_gold"],
            valid_cls["label_pred"],
            labels=["bug_report", "feature_request", "user_experience", "rating"],
            average="weighted",
            zero_division=0,
        ),
    }

    summary_eval_df, summary_overall_metrics, _ = evaluate_summaries(
        df,
        gold_col="summary_gold",
        pred_col="summary_pred",
        bert_model="xlm-roberta-base",
        batch_size=16,
    )

    cls_metrics["rougeL_f1"] = summary_overall_metrics.loc["rougeL_f1", "mean_value"]
    cls_metrics["bertscore_f1"] = summary_overall_metrics.loc["bertscore_f1", "mean_value"]

    return cls_metrics


all_model_metrics = []

for model_display_name, csv_path in prediction_files.items():
    if csv_path.exists():
        print("Evaluating:", model_display_name)
        all_model_metrics.append(evaluate_saved_prediction_file(model_display_name, csv_path))
    else:
        print("File not found:", csv_path)

llm_comparison_df = pd.DataFrame(all_model_metrics).sort_values(
    by=["macro_f1", "bertscore_f1"],
    ascending=False,
)

llm_comparison_path = RESULTS_DIR / "llm_models_comparison_metrics.csv"
llm_comparison_df.to_csv(llm_comparison_path, index=False)

print("Saved comparison to:", llm_comparison_path)
llm_comparison_df


Evaluating: Qwen3-4B-Instruct-2507


Calculating ROUGE-L:   0%|          | 0/480 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

  0%|          | 0/35 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/30 [00:00<?, ?it/s]

done in 0.22 seconds, 2135.33 sentences/sec
Saved comparison to: /content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference/llm_inference_results/llm_models_comparison_metrics.csv


,model,n_rows,parse_errors,accuracy,macro_f1,weighted_f1,rougeL_f1,bertscore_f1
0,Qwen3-4B-Instruct-2507,480,1,0.697286,0.671289,0.692033,0.203367,0.858441
1,Phi-4-mini-instruct,480,0,0.629167,0.592707,0.624514,0.160409,0.852721
2,Qwen2.5-7B-Instruct,480,1,0.567850,0.563870,0.521434,0.286550,0.874389
3,Mistral-7B-Instruct-v0.3,480,2,0.610879,0.540743,0.596179,0.118946,0.842042


In [20]:
# Пример: открыть конкретный CSV и посмотреть classification_report детально

model_to_inspect = "Qwen3-4B-Instruct-2507"
inspect_df = pd.read_csv(prediction_files[model_to_inspect])

valid_eval = inspect_df[inspect_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

print("Model:", model_to_inspect)
print("Parse errors:", (inspect_df["label_pred"] == "parse_error").sum())

print(classification_report(
    valid_eval["label_gold"],
    valid_eval["label_pred"],
    labels=["bug_report", "feature_request", "user_experience", "rating"],
    digits=4,
    zero_division=0,
))

inspect_df.head()


Model: Qwen3-4B-Instruct-2507
Parse errors: 1
                 precision    recall  f1-score   support

     bug_report     0.6069    0.8400    0.7047       125
feature_request     0.5000    0.5385    0.5185        26
user_experience     0.7925    0.5551    0.6528       227
         rating     0.7479    0.8812    0.8091       101

       accuracy                         0.6973       479
      macro avg     0.6618    0.7037    0.6713       479
   weighted avg     0.7188    0.6973    0.6920       479



,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,Сообщения не доходят и не видны,"{""label"": ""bug_report"", ""summary"": ""Сообщения ..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,bug_report,После удаления историй нельзя публиковать,"{""label"": ""bug_report"", ""summary"": ""После удал..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,Приложение виснит и вылетает при музыке,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Приложение после обновления не работает,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,user_experience,"Неудобное приложение, проблемы с использованием","{""label"": ""user_experience"", ""summary"": ""Неудо..."


In [23]:
summary_eval_df, summary_overall_metrics, _ = evaluate_summaries(
    inspect_df,
    gold_col="summary_gold",
    pred_col="summary_pred",
    bert_model="xlm-roberta-base",
    batch_size=16
)

summary_overall_metrics

Calculating ROUGE-L:   0%|          | 0/480 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/30 [00:00<?, ?it/s]

done in 0.99 seconds, 486.53 sentences/sec


,mean_value
rougeL_precision,0.216639
rougeL_recall,0.211860
rougeL_f1,0.203367
bertscore_precision,0.859104
bertscore_recall,0.858328
bertscore_f1,0.858441


In [ ]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
].sort_values("bertscore_f1").head(20)

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
96,"Вроде оф приложения, но такое не удобное. Ладн...",Добавить настройки сообщений,"{""label"": ""bug_report"", ""summary"": ""Приложение...",0.0,0.766506
212,бесполезно говорить,Краткая негативная оценка без деталей,Приложение бесполезно,0.0,0.772590
230,здравствуйте я скачала телеграм но мне не откр...,Недовольство платными функциями,Приложение не открывается из-за премиум-версии,0.0,0.779051
463,"серьёзно, мне вот интересно зачем вам добавлят...",Запретить ботам писать пользователю,Приложение не работает,0.0,0.783255
124,Приложение просто замечательнО! Но есть не бол...,Добавить галочки прочтения сообщений,Пользователь сообщает о невозможности получить...,0.0,0.783936
271,Отлично !,Краткая положительная оценка без деталей,Отлично,0.0,0.785634
447,спасибо,Краткая положительная оценка без деталей,Благодарит за приложение,0.0,0.786541
462,за всё спасибо,Краткая положительная оценка без деталей,Благодарит за приложение,0.0,0.786541
437,Все ОК!,Краткая положительная оценка без деталей,Все в порядке,0.0,0.787380
76,Персональные менеджеры авито полное дно. Разоб...,Недовольство модерацией и поддержкой,Персональные менеджеры Авито не работают,0.0,0.787533


In [ ]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
].sort_values("bertscore_f1", ascending=False).head(20)

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
250,ОФЛАЙН КАРТЫ НЕ РАБОТАЮТ !!! Скачаны 1200Мб мо...,ОФЛАЙН КАРТЫ НЕ РАБОТАЮТ,Офлайн карты не работают,1.000000,1.000000
166,добавьте подробную схему метро,Добавить подробную схему метро,Просьба добавить подробную схему метро,0.888889,0.971536
276,"без впн не работает, отстой",Приложение не работает без ВПН,Приложение не работает без VPN,0.800000,0.971042
79,вылетает постоянно,Приложение вылетает,Приложение вылетает постоянно,0.800000,0.967700
32,отлично!,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
421,молодец,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
48,высший класс,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
428,отличное,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
69,отличное,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
78,Имба!,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779


In [ ]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
][summary_eval_df["rougeL_f1"].between(0.6, 0.8)]

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
29,Уважаемые разработчики спасибо за приложение! ...,Проблемы с восстановлением аккаунта,Проблемы с безопасностью аккаунта и восстановл...,0.666667,0.936501
32,отлично!,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
36,"Уважаемая команда WhatsApp, мой WhatsApp не ра...",Проблемы с регистрацией,Пользователь описывает проблемы с регистрацией...,0.666667,0.892052
39,Приложение после обновления разочаровало из-за...,вернуть выбор способа передвижения,Пользователь просит вернуть выбор способа пере...,0.800000,0.944244
48,высший класс,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
49,"лагает максимально, смс не отображает",Приложение лагает и тормозит,Приложение лагает и не отображает SMS,0.600000,0.914139
69,отличное,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
78,Имба!,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
79,вылетает постоянно,Приложение вылетает,Приложение вылетает постоянно,0.800000,0.967700
97,Не работает,Приложение работает некорректно,Приложение не работает,0.666667,0.911029
